# Análise Exploratória e Financeira — TheLook eCommerce (BigQuery)

## 1. Contextualização e Escolha da Base de Dados
Para este projeto, foi utilizada a base de dados pública **`bigquery-public-data.thelook_ecommerce`**, disponível no Google BigQuery.

**Por que essa base foi escolhida?**
* **Estrutura relacional completa:** Simula um ecossistema real de comércio eletrônico com tabelas interligadas de pedidos (`orders`), itens vendidos (`order_items`), catálogo de produtos (`products`) e base de clientes (`users`).
* **Versatilidade analítica:** Permite responder tanto a perguntas operacionais e logísticas (prazos de entrega e funil de status de pedidos) quanto a análises financeiras e comerciais (faturamento mensal, ticket médio e margem bruta por categoria).
* **Controle de custos:** Como boa prática no ambiente BigQuery, todas as consultas foram precedidas por estimativas de processamento (`dry_run=True`) e limitadas por um teto de segurança (`maximum_bytes_billed = 1 GB`).

## 2. Exploração Inicial dos Dados
A partir da inspeção de metadados (`client.get_table`), identificamos a volumetria e a tipagem das quatro tabelas principais do dataset:

| Tabela | Total de Registros | Principais Colunas e Tipos |
| :--- | :--- | :--- |
| **`orders`** | 124.916 | `order_id` (`INTEGER`), `user_id` (`INTEGER`), `status` (`STRING`), `created_at`, `shipped_at`, `delivered_at`, `returned_at` (`TIMESTAMP`) |
| **`order_items`** | 181.396 | `id`, `order_id`, `product_id` (`INTEGER`), `status` (`STRING`), `sale_price` (`FLOAT`), datas em `TIMESTAMP` |
| **`products`** | 29.120 | `id` (`INTEGER`), `category`, `brand`, `department` (`STRING`), `cost` e `retail_price` (`FLOAT`) |
| **`users`** | 100.000 | `id`, `age` (`INTEGER`), `gender`, `city`, `country`, `traffic_source` (`STRING`), `user_geom` (`GEOGRAPHY`) |

**Observações sobre o ciclo de vida dos pedidos (`status`):**
Os 124.916 pedidos distribuem-se em cinco estados operacionais: **Shipped** (37.471), **Complete** (31.482), **Processing** (24.854), **Cancelled** (18.723) e **Returned** (12.386). Isso confirma que apenas uma parcela dos registros representa vendas efetivamente concluídas (`Complete`), exigindo filtragem explícita nas análises de receita e logística.

In [1]:
from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project="teste-v1-514421")

LIMITE_BYTES = 1 * 1024**3  # 1 GB: se a query passar disso, o BigQuery cancela

def estimar_mb(sql):
    """Estima quantos MB a query leria, sem executar (grátis)."""
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return client.query(sql, job_config=cfg).total_bytes_processed / 1024**2

def rodar(sql):
    """Executa a query com o teto de bytes ativado."""
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=LIMITE_BYTES)
    return client.query(sql, job_config=cfg).to_dataframe()

In [2]:
tabelas = ["orders", "order_items", "products", "users"]
for t in tabelas:
    tab = client.get_table(f"bigquery-public-data.thelook_ecommerce.{t}")  # só metadados
    print(f"\n{t} | linhas: {tab.num_rows}")
    for campo in tab.schema:
        print("   ", campo.name, campo.field_type)

# valores possíveis de status em orders
print(rodar("""
SELECT status, COUNT(*) AS total
FROM `bigquery-public-data.thelook_ecommerce.orders`
GROUP BY status
ORDER BY total DESC
"""))


orders | linhas: 124916
    order_id INTEGER
    user_id INTEGER
    status STRING
    gender STRING
    created_at TIMESTAMP
    returned_at TIMESTAMP
    shipped_at TIMESTAMP
    delivered_at TIMESTAMP
    num_of_item INTEGER

order_items | linhas: 181396
    id INTEGER
    order_id INTEGER
    user_id INTEGER
    product_id INTEGER
    inventory_item_id INTEGER
    status STRING
    created_at TIMESTAMP
    shipped_at TIMESTAMP
    delivered_at TIMESTAMP
    returned_at TIMESTAMP
    sale_price FLOAT

products | linhas: 29120
    id INTEGER
    cost FLOAT
    category STRING
    name STRING
    brand STRING
    retail_price FLOAT
    department STRING
    sku STRING
    distribution_center_id INTEGER

users | linhas: 100000
    id INTEGER
    first_name STRING
    last_name STRING
    email STRING
    age INTEGER
    gender STRING
    state STRING
    street_address STRING
    postal_code STRING
    city STRING
    country STRING
    latitude FLOAT
    longitude FLOAT
    traffic_s

## 3. Documentação Técnica e Interpretação das Consultas

### Query 1: Inspeção de Amostra da Tabela de Pedidos (`q01_amostra_pedidos`)
* **Objetivo:** Obter uma visão inicial da estrutura física e do preenchimento dos campos da tabela `orders` sem onerar o processamento.
* **Cláusulas SQL utilizadas:** `SELECT *` para retornar todas as colunas e `LIMIT 100` para restringir a amostra às primeiras 100 linhas.
* **Interpretação dos resultados:** As 100 primeiras linhas retornadas correspondem a pedidos com status `Cancelled` realizados entre 2022 e 2023. Como esperado para pedidos cancelados, as colunas `shipped_at`, `delivered_at` e `returned_at` apresentam 100% de valores nulos (`NaT`), confirmando que a ausência de datas logísticas está coerente com a regra de negócio.

---

### Query 2: Pedidos Concluídos com Maior Tempo de Entrega (`q02_prazo_entrega_pedidos`)
* **Objetivo:** Identificar os 50 pedidos concluídos mais recentes que apresentaram o maior prazo entre a criação do pedido e a entrega ao cliente a partir de 2024.
* **Cláusulas SQL utilizadas:**
  * `WHERE`: Filtra apenas pedidos com `status = 'Complete'`, data de entrega preenchida (`IS NOT NULL`) e criados a partir de `2024-01-01`.
  * `TIMESTAMP_DIFF(..., DAY)`: Calcula a diferença em dias inteiros entre `created_at` e `delivered_at`.
  * `ORDER BY ... DESC` e `LIMIT 50`: Ordena pelos maiores prazos de entrega e desempata pelas datas mais recentes.
* **Interpretação dos resultados:** Os 50 registros retornados atingiram o teto máximo de **7 dias** para entrega, concentrados em pedidos realizados em novembro de 2026. Isso indica que a operação logística da base sintética possui um limite superior bem definido de até uma semana para conclusão da entrega.

---

### Query 3: Evolução Mensal de Receita e Volume por Status (`q03_receita_mensal_status`)
* **Objetivo:** Acompanhar a evolução mensal do volume de itens, pedidos únicos, receita bruta e preço médio por item desde janeiro de 2024, segmentados pelo status do item.
* **Cláusulas SQL utilizadas:**
  * `DATE_TRUNC(DATE(created_at), MONTH)`: Agrupa os carimbos de data/hora no primeiro dia de cada mês.
  * `COUNT(*)`, `COUNT(DISTINCT order_id)`, `SUM()` e `AVG()`: Funções de agregação para contabilizar itens, pedidos distintos, faturamento e ticket médio por item.
  * `GROUP BY` e `ORDER BY`: Consolida as métricas por mês e status em ordem cronológica decrescente.
* **Interpretação dos resultados:** A consulta retornou 175 combinações de mês/status entre janeiro de 2024 e novembro de 2026. O volume e a receita mostram crescimento consistente ao longo do período, atingindo o pico em **novembro de 2026**, quando os itens em trânsito (`Shipped`) somaram **USD 190.685,12** (3.176 itens) e os concluídos (`Complete`) atingiram **USD 166.420,59** (2.691 itens), oscilando entre USD 53,15 e USD 67,24 ao longo do período (e entre **USD 58,21 e USD 62,55** em novembro/2026).

---

### Query 4: Desempenho Financeiro e Margem Bruta por Categoria (`q04_margem_por_categoria`)
* **Objetivo:** Cruzar os itens efetivamente vendidos (`Complete`) a partir de 2025 com o cadastro de produtos para avaliar quais categorias geram maior receita e maior rentabilidade.
* **Cláusulas SQL utilizadas:**
  * `INNER JOIN`: Relaciona `order_items` (`oi`) com `products` (`p`) através da chave `oi.product_id = p.id`.
  * Cálculos derivados e `SAFE_DIVIDE`: Calcula a margem bruta absoluta (`sale_price - cost`) e a margem percentual sobre a receita evitando erros de divisão por zero.
* **Interpretação dos resultados:** Das 26 categorias analisadas:
  * **Liderança em faturamento:** *Outerwear & Coats* lidera a receita total com **USD 208.152,26** (1.368 itens e margem de 55,5%), seguida por *Jeans* com **USD 190.679,06** (maior volume físico, com 1.907 itens, mas margem menor de 46,5%) e *Sweaters* com **USD 116.215,23**.
  * **Maior rentabilidade percentual:** *Blazers & Jackets* destaca-se com **62,0%** de margem bruta (USD 46.488,73 de receita), seguida por *Socks & Hosiery* (60,1%) e *Skirts* (60,1%).
  * **Menor rentabilidade percentual:** *Clothing Sets* apresenta a menor margem percentual (**36,9%**), indicando oportunidade de revisão de precificação ou custo de fornecimento.

In [3]:
from datetime import datetime
print("Executado em:", datetime.now().strftime("%d/%m/%Y %H:%M"))

q01_amostra_pedidos = """
SELECT *
FROM `bigquery-public-data.thelook_ecommerce.orders`
LIMIT 100
"""

q02_prazo_entrega_pedidos = """
SELECT
  order_id,
  user_id,
  status,
  created_at,
  delivered_at,
  TIMESTAMP_DIFF(delivered_at, created_at, DAY) AS dias_ate_entrega
FROM `bigquery-public-data.thelook_ecommerce.orders`
WHERE status = 'Complete'
  AND delivered_at IS NOT NULL
  AND created_at >= TIMESTAMP('2024-01-01')
ORDER BY dias_ate_entrega DESC, created_at DESC
LIMIT 50
"""

q03_receita_mensal_status = """
SELECT
  DATE_TRUNC(DATE(created_at), MONTH) AS mes,
  status,
  COUNT(*)                            AS total_itens,
  COUNT(DISTINCT order_id)            AS total_pedidos,
  ROUND(SUM(sale_price), 2)           AS receita_bruta,
  ROUND(AVG(sale_price), 2)           AS preco_medio_item
FROM `bigquery-public-data.thelook_ecommerce.order_items`
WHERE created_at >= TIMESTAMP('2024-01-01')
GROUP BY mes, status
ORDER BY mes DESC, status
"""

q04_margem_por_categoria = """
SELECT
  p.category                                   AS categoria,
  COUNT(*)                                     AS itens_vendidos,
  ROUND(SUM(oi.sale_price), 2)                 AS receita,
  ROUND(SUM(oi.sale_price - p.cost), 2)        AS margem_bruta,
  ROUND(SAFE_DIVIDE(SUM(oi.sale_price - p.cost),
                    SUM(oi.sale_price)) * 100, 1) AS margem_pct
FROM `bigquery-public-data.thelook_ecommerce.order_items` AS oi
INNER JOIN `bigquery-public-data.thelook_ecommerce.products` AS p
  ON oi.product_id = p.id
WHERE oi.status = 'Complete'
  AND oi.created_at >= TIMESTAMP('2025-01-01')
GROUP BY categoria
ORDER BY receita DESC
LIMIT 100
"""

queries = {
    "q01_amostra_pedidos": q01_amostra_pedidos,
    "q02_prazo_entrega_pedidos": q02_prazo_entrega_pedidos,
    "q03_receita_mensal_status": q03_receita_mensal_status,
    "q04_margem_por_categoria": q04_margem_por_categoria,
}

for nome, sql in queries.items():
    print(nome, "-> leria aprox.", round(estimar_mb(sql), 1), "MB")

Executado em: 16/11/2026 17:15
q01_amostra_pedidos -> leria aprox. 6.4 MB
q02_prazo_entrega_pedidos -> leria aprox. 4.4 MB
q03_receita_mensal_status -> leria aprox. 5.9 MB
q04_margem_por_categoria -> leria aprox. 6.7 MB


## 4. Validação de Consistência e Qualidade dos Dados
Antes de exportar os resultados finais para CSV, foram executadas verificações de integridade tanto no BigQuery quanto nos DataFrames em Python:

1. **Unicidade de chaves (`duplicidade_order_items`):** A comparação entre `COUNT(*)` e `COUNT(DISTINCT id)` na tabela `order_items` retornou exatamente **181.396** registros em ambos, comprovando ausência de linhas duplicadas.
2. **Integridade referencial (`itens_sem_produto`):** O cruzamento via `LEFT JOIN` entre `order_items` e `products` apontou **0 itens órfãos**, garantindo que o `INNER JOIN` da Query 4 não descartou nenhuma venda legítima.
3. **Coerência de nulos (`nulos_em_entregues` e `df.isna().sum()`):** Todos os **31.482** pedidos com `status = 'Complete'` possuem data de entrega (`delivered_at`) preenchida. Nos DataFrames exportados das Queries 2, 3 e 4, não há nenhum valor nulo (`0` em todas as colunas); na Query 1, os nulos restritos às colunas logísticas ocorrem exclusivamente porque a amostra contém apenas pedidos cancelados.

In [4]:
checagens = {
    "duplicidade_order_items": """
        SELECT COUNT(*) AS linhas, COUNT(DISTINCT id) AS ids_unicos
        FROM `bigquery-public-data.thelook_ecommerce.order_items`
    """,
    "itens_sem_produto": """
        SELECT COUNT(*) AS itens_sem_produto
        FROM `bigquery-public-data.thelook_ecommerce.order_items` AS oi
        LEFT JOIN `bigquery-public-data.thelook_ecommerce.products` AS p
          ON oi.product_id = p.id
        WHERE p.id IS NULL
    """,
    "nulos_em_entregues": """
        SELECT
          COUNT(*) AS pedidos_complete,
          COUNTIF(delivered_at IS NULL) AS sem_data_entrega
        FROM `bigquery-public-data.thelook_ecommerce.orders`
        WHERE status = 'Complete'
    """,
}

for nome, sql in checagens.items():
    print(f"\n{nome}")
    print(rodar(sql))


duplicidade_order_items
   linhas  ids_unicos
0  181396      181396

itens_sem_produto
   itens_sem_produto
0                  0

nulos_em_entregues
   pedidos_complete  sem_data_entrega
0             31482                 0


In [5]:
PASTA = Path(r"C:\Users\Usuario\Desktop\EBAC\the_eco")
PASTA.mkdir(parents=True, exist_ok=True)



resultados = {}
for nome, sql in queries.items():
    df = rodar(sql)
    resultados[nome] = df
    print(f"\n{nome}: {len(df)} linhas | nulos por coluna:")
    print(df.isna().sum().to_string())
    print(df.head())
    df.to_csv(PASTA / f"{nome}.csv",
              index=False, encoding="utf-8")


q01_amostra_pedidos: 100 linhas | nulos por coluna:
order_id          0
user_id           0
status            0
gender            0
created_at        0
returned_at     100
shipped_at      100
delivered_at    100
num_of_item       0
   order_id  user_id     status gender                created_at returned_at  \
0         1        1  Cancelled      F 2022-09-14 11:01:54+00:00         NaT   
1         2        1  Cancelled      F 2023-07-06 07:00:43+00:00         NaT   
2         3        1  Cancelled      F 2023-06-12 08:38:12+00:00         NaT   
3         6        5  Cancelled      F 2022-11-07 02:30:28+00:00         NaT   
4         7        6  Cancelled      F 2022-05-29 12:51:34+00:00         NaT   

  shipped_at delivered_at  num_of_item  
0        NaT          NaT            1  
1        NaT          NaT            2  
2        NaT          NaT            1  
3        NaT          NaT            1  
4        NaT          NaT            3  

q02_prazo_entrega_pedidos: 50 linhas | 